In [10]:
import pandas as pd
import numpy as np

# 1. Load embeddings dan hasil penugasan topic
df_emb = pd.read_csv('embedding_bertopic_v2.csv')              # kolom 'Game', 'cleaned_Reviews', embedding_*
df_asg = pd.read_csv('hasil_topic_assignment_automerged.csv')  # kolom 'game', 'cleaned_reviews', 'topic', UMAP_*, dll.

# 2. Merge berdasarkan teks review (perhatikan perbedaan kapitalisasi kolom)
df = pd.merge(
    df_emb,
    df_asg[['cleaned_reviews', 'topic']],
    left_on='cleaned_Reviews',
    right_on='cleaned_reviews',
    how='inner'
)

# (Opsional) Drop kolom duplikat
df = df.drop(columns=['cleaned_reviews'])

# 3. Tentukan kolom embedding penuh secara otomatis
emb_cols = [c for c in df.columns if c.startswith('embedding_')]

# 4. Hitung centroid rata-rata per topic, lalu normalisasi jadi unit-vector
centroids = df.groupby('topic')[emb_cols].mean()
centroids = centroids.div(np.linalg.norm(centroids, axis=1), axis=0)

# 5. Fungsi cosine similarity (dot product karena vektor sudah unit-length)
def cosine_sim(a, b):
    return np.dot(a, b)

# 6. Normalisasi embedding tiap review & hitung similarity ke centroid topiknya
df['similarity'] = df.apply(
    lambda r: cosine_sim(
        r[emb_cols].values / np.linalg.norm(r[emb_cols].values),
        centroids.loc[r['topic']].values
    ),
    axis=1
)

# 7. Rata-rata similarity per (Game, topic)
gt_sim = (
    df
    .groupby(['Game', 'topic'])['similarity']
    .mean()
    .reset_index()
)

# 8. Pilih topic dominan per Game
idx = gt_sim.groupby('Game')['similarity'].idxmax()
dominant = gt_sim.loc[idx].reset_index(drop=True)

# 9. Rename kolom dan simpan hasil akhir
dominant = dominant.rename(columns={
    'Game': 'Game',
    'topic': 'Dominant_Topic',
    'similarity': 'Similarity_Score'
})
dominant.to_csv('dominant_topic_per_game.csv', index=False)

print(dominant.head())


                   Game  Dominant_Topic  Similarity_Score
0         7 Days to Die              16          0.778779
1          A Short Hike               5          0.708093
2             A Way Out              21          0.727978
3             ASTRONEER               7          0.709343
4  AdVenture Capitalist               1          0.556531


In [9]:
import pandas as pd

df_emb = pd.read_csv('embedding_bertopic_v2.csv')
df_asg = pd.read_csv('hasil_topic_assignment_automerged.csv')

print("Embedding file columns:\n", df_emb.columns.tolist())
print("Topic-assignment file columns:\n", df_asg.columns.tolist())

Embedding file columns:
 ['Game', 'cleaned_Reviews', 'num_words', 'review_length', 'embedding_0', 'embedding_1', 'embedding_2', 'embedding_3', 'embedding_4', 'embedding_5', 'embedding_6', 'embedding_7', 'embedding_8', 'embedding_9', 'embedding_10', 'embedding_11', 'embedding_12', 'embedding_13', 'embedding_14', 'embedding_15', 'embedding_16', 'embedding_17', 'embedding_18', 'embedding_19', 'embedding_20', 'embedding_21', 'embedding_22', 'embedding_23', 'embedding_24', 'embedding_25', 'embedding_26', 'embedding_27', 'embedding_28', 'embedding_29', 'embedding_30', 'embedding_31', 'embedding_32', 'embedding_33', 'embedding_34', 'embedding_35', 'embedding_36', 'embedding_37', 'embedding_38', 'embedding_39', 'embedding_40', 'embedding_41', 'embedding_42', 'embedding_43', 'embedding_44', 'embedding_45', 'embedding_46', 'embedding_47', 'embedding_48', 'embedding_49', 'embedding_50', 'embedding_51', 'embedding_52', 'embedding_53', 'embedding_54', 'embedding_55', 'embedding_56', 'embedding_57',

In [13]:
import pandas as pd
import numpy as np
from collections import Counter

# 1. Load data
df_dom = pd.read_csv("dominant_topic_per_game.csv")   # Game, Dominant_Topic, Similarity_Score
df_seg = pd.read_csv("sampled_segmentation.csv")     # Steam ID, Game Name, Playtime (hours), Genres, Achievements

# 2. Standardisasi nama game agar cocok
df_seg["Game Name"] = df_seg["Game Name"].str.strip().str.lower()
df_dom["Game"]       = df_dom["Game"].str.strip().str.lower()

# 3. Merge untuk mendapatkan Dominant_Topic per baris game–pemain
merged = pd.merge(
    df_seg, df_dom[["Game","Dominant_Topic"]],
    left_on="Game Name", right_on="Game",
    how="inner"
)

# 4. Filter invalid dan hanya pemain yang main ≥10 menit
merged = merged[merged["Game Name"] != "unknown"]
merged = merged[merged["Playtime (hours)"] >= 0.167]

# 5. Buang NaN & duplikat
merged.dropna(subset=[
    "Steam ID","Game Name","Playtime (hours)",
    "Genres","Achievements","Dominant_Topic"
], inplace=True)
merged.drop_duplicates(subset=["Steam ID","Game Name"], inplace=True)

# 6. Hanya pemain aktif (>= kuartil 1 total game)
game_counts = (
    merged
    .groupby("Steam ID")["Game Name"]
    .nunique()
    .reset_index(name="Total_Games")
)
q1 = game_counts["Total_Games"].quantile(0.25)
active_ids = game_counts[game_counts["Total_Games"] >= q1]["Steam ID"]
merged = merged[merged["Steam ID"].isin(active_ids)]

# 7. Hitung Total_Achievements & Avg_Playtime
achievement_sum = (
    merged
    .groupby("Steam ID")["Achievements"]
    .sum()
    .reset_index(name="Total_Achievements")
)
playtime_avg = (
    merged
    .groupby("Steam ID")["Playtime (hours)"]
    .mean()
    .reset_index(name="Avg_Playtime")
)

# 8. Mapping Genres → kode numerik
merged["Genres"] = merged["Genres"].astype(str)
genre_list = sorted({g for gs in merged["Genres"] for g in gs.split(", ")})
genre_map  = {g:i+1 for i,g in enumerate(genre_list)}
merged["Genre_Code"] = merged["Genres"].apply(
    lambda x: [genre_map[g] for g in x.split(", ") if g in genre_map]
)

# 9. Buat fungsi top_n dan ambil Top 3 Genre per pemain
def top_n(seq, n=3):
    c = Counter(seq)
    return [item for item, _ in c.most_common(n)]

top_genres = (
    merged
    .groupby("Steam ID")["Genre_Code"]
    .apply(lambda lists: top_n([g for sub in lists for g in sub], 3))
    .reset_index(name="Top_3_Genres")
)

# 10. Ambil Dominant_Topic bersih per pemain
topic_lists = (
    merged
    .groupby("Steam ID")["Dominant_Topic"]
    .apply(lambda ts: [t for t,_ in Counter(ts).most_common()])
)
def choose_topic(lst):
    for t in lst:
        if t != -1:
            return t
    return np.nan
dominant_topic = (
    topic_lists
    .apply(choose_topic)
    .reset_index(name="Topic Dominan")
)

# 11. Gabungkan semua ke summary
summary = (
    game_counts
    .merge(achievement_sum, on="Steam ID")
    .merge(playtime_avg,    on="Steam ID")
    .merge(dominant_topic,  on="Steam ID")
    .merge(top_genres,      on="Steam ID")
)

# 12. Explode Top_3_Genres → satu baris per genre dominan
exploded = (
    summary
    .explode("Top_3_Genres")
    .rename(columns={
        "Steam ID":         "Steam ID",
        "Total_Games":      "Total Game",
        "Avg_Playtime":     "Total Playtime",
        "Total_Achievements":"Total Achievement",
        "Top_3_Genres":     "Genre Dominan"
    })
)

# 13. Tambah kolom No
exploded.insert(0, "No", range(1, len(exploded) + 1))

# 14. Simpan hasil
exploded.to_csv("transformation_segmentation_v4.csv", index=False)
print("✅ Selesai. Hasil dengan Top 3 Genre explode tersimpan di 'transformation_segmentation_v4.csv'")


✅ Selesai. Hasil dengan Top 3 Genre explode tersimpan di 'transformation_segmentation_v4.csv'


In [1]:
import pandas as pd

df = pd.read_csv("transformation_segmentation_v4.csv")

df.to_excel("transformation_segmentation_v4.xlsx")